In [11]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.notebook import tqdm
import emoji


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ Running on CPU")


model_path = "model_path"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

model.to(device)

# ✅ Faster inference on GPU
if device.type == "cuda":
    model = model.half()
    torch.backends.cudnn.benchmark = True

model.eval()

print("✅ Model loaded!")


df = pd.read_csv(
    "load_data.csv  
)

df = df.rename(columns={"comment": "text"})

df = df[df["text"].notna()]
df = df[df["text"].astype(str).str.strip() != ""]
df = df[df["text"].astype(str).str.len() > 2]

print("✅ Data loaded:", df.shape)


COURSE_CONTEXT = (
    "Write your context"
)

df["text"] = df["text"].astype(str)

# ✅ emoji to text
df["text"] = df["text"].apply(lambda x: emoji.demojize(x))

# ✅ add domain context
df["text"] = COURSE_CONTEXT + df["text"]

texts = df["text"].tolist()


def run_model(texts):

    batch_size = 128 if device.type == "cpu" else 256

    predictions = []
    confidences = []

    label_map = {0: "positive", 1: "neutral", 2: "negative"}

    for i in tqdm(range(0, len(texts), batch_size), desc="🚀 Running Model"):

        batch = texts[i:i + batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors="pt"
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():

            # ✅ Mixed precision (faster)
            if device.type == "cuda":
                with torch.cuda.amp.autocast():
                    outputs = model(**inputs)
            else:
                outputs = model(**inputs)

            probs = F.softmax(outputs.logits, dim=1)
            preds = torch.argmax(probs, dim=1)

        # ✅ confidence (gap)
        top2 = torch.topk(probs, 2, dim=1).values
        conf = (top2[:, 0] - top2[:, 1]).cpu().numpy()

        # ✅ normalize
        conf = (conf - conf.min()) / (conf.max() - conf.min() + 1e-8)

        predictions.extend(preds.cpu().numpy())
        confidences.extend(conf)

    labels = [label_map[p] for p in predictions]

    return labels, confidences


print("\n🚀 Starting inference...\n")

labels, confidences = run_model(texts)

df["sentiment"] = labels
df["confidence"] = confidences

print("\n✅ Inference complete!")


positive_pct = (df["sentiment"] == "positive").mean() * 100
negative_pct = (df["sentiment"] == "negative").mean() * 100
avg_conf = df["confidence"].mean()

recommendation_score = (
    positive_pct * 1.0 +
    (100 - negative_pct) * 0.5
) / 1.5

print("\n📊 METRICS")
print("Total comments:", len(df))
print("Positive %:", round(positive_pct, 2))
print("Negative %:", round(negative_pct, 2))
print("Avg confidence:", round(avg_conf, 3))
print("Recommendation score:", round(recommendation_score, 2))


print("\n🎯 RECOMMENDATION")

if recommendation_score > 70:
    print("✅ Highly Recommended")
elif recommendation_score > 40:
    print("✅ Recommended")
elif recommendation_score > 10:
    print("⚠️ Average")
else:
    print("❌ Needs Improvement")


output_path = "output_path"

df.to_csv(output_path, index=False)

print("\n✅ Saved predictions to:", output_path)


print("\n🔍 Sample predictions:\n")
display(df.head())

Using device: cuda
GPU: Tesla T4


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Model loaded!
✅ Data loaded: (26801, 2)

🚀 Starting inference...



🚀 Running Model:   0%|          | 0/105 [00:00<?, ?it/s]

/tmp/ipykernel_57/83400823.py:102: FutureWarning:

`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.




✅ Inference complete!

📊 METRICS
Total comments: 26801
Positive %: 75.94
Negative %: 11.36
Avg confidence: 0.991
Recommendation score: 80.17

🎯 RECOMMENDATION
✅ Highly Recommended

✅ Saved predictions to: /kaggle/working/predicted_output.csv

🔍 Sample predictions:



/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning:

overflow encountered in cast



,content_id,text,sentiment,confidence
0,do_1143414519384227841585,This comment is about a government learning co...,positive,1.000000
1,do_1142826090669342721118,This comment is about a government learning co...,positive,1.000000
3,do_1142989662796595201124,This comment is about a government learning co...,positive,1.000000
4,do_1140017961653534721223,This comment is about a government learning co...,positive,1.000000
5,do_1144114748852305921561,This comment is about a government learning co...,neutral,0.998047


In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.notebook import tqdm
import plotly.express as px
from collections import Counter
import emoji

# ======================================
# DEVICE
# ======================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ======================================
# LOAD MODEL
# ======================================
model_path = "model.path"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

model.to(device)
model.eval()

print("✅ Model loaded!")

# ======================================
# LOAD DATA
# ======================================
df = pd.read_csv("data.csv")

df = df.rename(columns={"comment": "text"})
df = df[df["text"].notna()]

df["text"] = df["text"].astype(str).apply(lambda x: emoji.demojize(x))

texts = df["text"].tolist()

print("✅ Data ready:", df.shape)

# ======================================
# MODEL INFERENCE
# ======================================
def run_model(texts):
    batch_size = 128 if device.type == "cpu" else 256

    predictions = []
    confidences = []

    label_map = {0: "positive", 1: "neutral", 2: "negative"}

    for i in tqdm(range(0, len(texts), batch_size), desc="Running Model"):

        batch = texts[i:i + batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors="pt"
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

            probs = F.softmax(outputs.logits, dim=1)
            preds = torch.argmax(probs, dim=1)

        top2 = torch.topk(probs, 2, dim=1).values
        conf = (top2[:, 0] - top2[:, 1]).cpu().numpy()

        conf = (conf - conf.min()) / (conf.max() - conf.min() + 1e-8)

        predictions.extend(preds.cpu().numpy())
        confidences.extend(conf)

    labels = [label_map[p] for p in predictions]
    return labels, confidences

# ======================================
# RUN INFERENCE
# ======================================
labels, confidences = run_model(texts)

df["sentiment"] = labels
df["confidence"] = confidences

print("✅ Inference complete!")

# ======================================
# CLI GRAPHIC WALKER
# ======================================
import re

class CLIWalker:

    def __init__(self, df):
        self.original_df = df.copy()
        self.df = df.copy()

    # ======================================
    # NATURAL LANGUAGE PARSER ✅
    # ======================================
    def interpret(self, cmd):

        cmd = cmd.lower()

        # sentiment queries
        if "negative" in cmd:
            self.df = self.df[self.df["sentiment"] == "negative"]
            return "✅ Showing negative comments"

        if "positive" in cmd:
            self.df = self.df[self.df["sentiment"] == "positive"]
            return "✅ Showing positive comments"

        if "neutral" in cmd:
            self.df = self.df[self.df["sentiment"] == "neutral"]
            return "✅ Showing neutral comments"

        # confidence filters
        if "high confidence" in cmd:
            self.df = self.df[self.df["confidence"] > 0.75]
            return "✅ High confidence data filtered"

        if "low confidence" in cmd:
            self.df = self.df[self.df["confidence"] < 0.3]
            return "✅ Low confidence data filtered"

        # plot commands
        if "plot sentiment" in cmd:
            return "plot sentiment"

        if "confidence distribution" in cmd:
            return "plot histogram"

        if "box plot" in cmd:
            return "plot box"

        if "pie" in cmd:
            return "plot pie"

        if "scatter" in cmd:
            return "plot scatter"

        if "trend" in cmd:
            return "plot trend"

        return None

    # ======================================
    # HELP MENU ✅
    # ======================================
    def help(self):
        print("""
📊 CLI WALKER COMMANDS

General:
- help
- exit
- reset
- export

Data:
- show columns
- show head
- show shape

Filtering:
- filter sentiment=positive
- filter confidence>0.5

Natural language:
- show negative comments
- show high confidence data
- plot sentiment distribution
- show sentiment trend

Charts:
- plot sentiment
- plot histogram
- plot pie
- plot box
- plot scatter
- plot trend

Analysis:
- top words
- show negative
""")

    # ======================================
    # MAIN LOOP ✅
    # ======================================
    def run(self):

        print("\n🚀 CLI Walker Started (type 'help')\n")

        while True:

            cmd = input(">> ").strip()

            if cmd == "exit":
                print("✅ Exiting")
                break

            # ✅ NATURAL LANGUAGE FIRST
            interpreted = self.interpret(cmd)

            if interpreted:
                if "✅" in interpreted:
                    print(interpreted)
                    continue
                else:
                    cmd = interpreted

            # =============================
            # GENERAL
            # =============================
            if cmd == "help":
                self.help()

            elif cmd == "reset":
                self.df = self.original_df.copy()
                print("✅ Data reset")

            elif cmd == "export":
                path = "/kaggle/working/filtered_output.csv"
                self.df.to_csv(path, index=False)
                print(f"✅ Exported to {path}")

            elif cmd == "show columns":
                print(self.df.columns.tolist())

            elif cmd == "show head":
                print(self.df.head())

            elif cmd == "show shape":
                print(self.df.shape)

            # =============================
            # FILTER
            # =============================
            elif cmd.startswith("filter"):
                try:
                    if "=" in cmd:
                        key, value = cmd.replace("filter ", "").split("=")
                        self.df = self.df[self.df[key.strip()] == value.strip()]

                    elif ">" in cmd:
                        key, value = cmd.replace("filter ", "").split(">")
                        self.df = self.df[self.df[key.strip()] > float(value.strip())]

                    elif "<" in cmd:
                        key, value = cmd.replace("filter ", "").split("<")
                        self.df = self.df[self.df[key.strip()] < float(value.strip())]

                    print(f"✅ Filtered: {len(self.df)} rows")

                except:
                    print("❌ Invalid filter")

            # =============================
            # AGGREGATION
            # =============================
            elif cmd == "group sentiment":
                print(self.df["sentiment"].value_counts())

            elif cmd == "avg confidence":
                print("Avg confidence:", self.df["confidence"].mean())

            # =============================
            # PLOTTING ✅
            # =============================
            elif cmd == "plot sentiment":
                temp = self.df["sentiment"].value_counts().reset_index()
                temp.columns = ["sentiment", "count"]
                px.bar(temp, x="sentiment", y="count", color="sentiment").show()

            elif cmd == "plot histogram":
                px.histogram(self.df, x="confidence", nbins=30).show()

            elif cmd == "plot pie":
                px.pie(self.df, names="sentiment").show()

            elif cmd == "plot box":
                px.box(self.df, x="sentiment", y="confidence").show()

            elif cmd == "plot scatter":
                temp = self.df.reset_index()
                px.scatter(temp, x=temp.index, y="confidence", color="sentiment").show()

            # ======================================
            # ✅ NEW: SENTIMENT TREND OVER TIME
            # ======================================
            elif cmd == "plot trend":

                if "timestamp" not in self.df.columns:
                    print("⚠️ No 'timestamp' column found in data")
                    continue

                temp = self.df.copy()
                temp["timestamp"] = pd.to_datetime(temp["timestamp"])

                trend = temp.groupby(
                    [pd.Grouper(key="timestamp", freq="D"), "sentiment"]
                ).size().reset_index(name="count")

                fig = px.line(
                    trend,
                    x="timestamp",
                    y="count",
                    color="sentiment",
                    title="Sentiment Trend Over Time"
                )

                fig.show()

            # =============================
            # ANALYSIS
            # =============================
            elif cmd == "top words":
                words = " ".join(self.df["text"]).split()
                print(Counter(words).most_common(10))

            elif cmd == "show negative":
                print(self.df[self.df["sentiment"] == "negative"].head())

            else:
                print("❌ Unknown command")
df["timestamp"] = pd.date_range(
    start="2024-01-01",
    periods=len(df),
    freq="H"   
)

Using device: cuda


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Model loaded!


UnicodeDecodeError: 'utf-8' codec can't decode bytes in position 7-8: invalid continuation byte

In [25]:
walker = CLIWalker(df)
walker.run()


🚀 CLI Walker Started (type 'help')



>>  help



📊 CLI WALKER COMMANDS

General:
- help
- exit
- reset
- export

Data:
- show columns
- show head
- show shape

Filtering:
- filter sentiment=positive
- filter confidence>0.5

Natural language:
- show negative comments
- show high confidence data
- plot sentiment distribution
- show sentiment trend

Charts:
- plot sentiment
- plot histogram
- plot pie
- plot box
- plot scatter
- plot trend

Analysis:
- top words
- show negative



>>  plot boxe


❌ Unknown command


...

>>  plot trend


>>  export


✅ Exported to /kaggle/working/filtered_output.csv


>>  exit


✅ Exiting


In [6]:
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.notebook import tqdm
import emoji
import hashlib
import os


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


model_path = "model.path"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

model.to(device)
model.eval()

print("✅ Model loaded!")

# ======================================
# LOAD DATA
# ======================================
def load_data(path):
    if path.endswith(".csv"):
        return pd.read_csv(path)
    elif path.endswith(".parquet"):
        return pd.read_parquet(path)
    elif path.endswith(".json"):
        return pd.read_json(path)
    else:
        raise ValueError("Unsupported file format")

file_path = "data.csv"
df = load_data(file_path)

# normalize column
df.columns = [c.lower() for c in df.columns]

if "comment" in df.columns:
    df = df.rename(columns={"comment": "text"})

df = df[df["text"].notna()]

# preprocessing
df["text"] = df["text"].astype(str).apply(emoji.demojize)

# timestamp (if not present)
if "timestamp" not in df.columns:
    df["timestamp"] = pd.date_range(
        start="2024-01-01",
        periods=len(df),
        freq="h"
    )

print("✅ Data ready:", df.shape)


def get_hash(text):
    return hashlib.md5(text.encode()).hexdigest()

df["row_id"] = df["text"].apply(get_hash)


cache_path = "/kaggle/working/cache_predictions.csv"

if os.path.exists(cache_path):
    cache_df = pd.read_csv(cache_path)
    print("✅ Cache loaded:", cache_df.shape)
else:
    cache_df = pd.DataFrame(columns=["row_id", "sentiment", "confidence"])
    print("⚠️ No cache found")


def run_model(texts):

    batch_size = 128 if device.type == "cpu" else 256

    predictions, confidences = [], []

    label_map = {0: "positive", 1: "neutral", 2: "negative"}

    for i in tqdm(range(0, len(texts), batch_size), desc="Running Model"):

        batch = texts[i:i + batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors="pt"
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            probs = F.softmax(outputs.logits, dim=1)
            preds = torch.argmax(probs, dim=1)

        top2 = torch.topk(probs, 2, dim=1).values
        conf = (top2[:, 0] - top2[:, 1]).cpu().numpy()

        conf = (conf - conf.min()) / (conf.max() - conf.min() + 1e-8)

        predictions.extend(preds.cpu().numpy())
        confidences.extend(conf)

    labels = [label_map[p] for p in predictions]
    return labels, confidences


cached_ids = set(cache_df["row_id"])

df_new = df[~df["row_id"].isin(cached_ids)]
print("🆕 New rows:", len(df_new))

if len(df_new) > 0:

    texts_new = df_new["text"].tolist()

    labels, confidences = run_model(texts_new)

    df_new["sentiment"] = labels
    df_new["confidence"] = confidences

    new_cache = df_new[["row_id", "sentiment", "confidence"]]

    cache_df = pd.concat([cache_df, new_cache], ignore_index=True)
    cache_df = cache_df.drop_duplicates(subset=["row_id"])

    cache_df.to_csv(cache_path, index=False)

    print("✅ Cache updated")

else:
    print("✅ Using cached results")

# ======================================
# MERGE RESULTS
# ======================================
df = df.merge(cache_df, on="row_id", how="left")

print("✅ Inference complete!")

# ======================================
# ✅ METRICS & RECOMMENDATION
# ======================================
positive_pct = (df["sentiment"] == "positive").mean() * 100
negative_pct = (df["sentiment"] == "negative").mean() * 100
neutral_pct = (df["sentiment"] == "neutral").mean() * 100
avg_conf = df["confidence"].mean()

# recommendation formula
recommendation_score = (
    positive_pct * 1.0 +
    (100 - negative_pct) * 0.5
) / 1.5

# recommendation label
if recommendation_score > 70:
    recommendation = "✅ Highly Recommended"
elif recommendation_score > 40:
    recommendation = "✅ Recommended"
elif recommendation_score > 10:
    recommendation = "⚠️ Average"
else:
    recommendation = "❌ Needs Improvement"

# ======================================
# ✅ PRINT SUMMARY
# ======================================
print("\n📊 SUMMARY\n")
print(f"Total comments: {len(df)}")
print(f"Positive: {positive_pct:.2f}%")
print(f"Neutral: {neutral_pct:.2f}%")
print(f"Negative: {negative_pct:.2f}%")
print(f"Average Confidence: {avg_conf:.3f}")
print(f"\n🎯 Recommendation Score: {recommendation_score:.2f}")
print(f"👉 {recommendation}")

# ======================================
# SAVE OUTPUT
# ======================================
output_path = "/kaggle/working/final_output.csv"
df.to_csv(output_path, index=False)

print(f"\n✅ Output saved to: {output_path}")


Using device: cuda


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Model loaded!
✅ Data ready: (29993, 3)
✅ Cache loaded: (39138, 3)
🆕 New rows: 0
✅ Using cached results
✅ Inference complete!

📊 SUMMARY

Total comments: 29993
Positive: 70.23%
Neutral: 19.70%
Negative: 10.07%
Average Confidence: 0.976

🎯 Recommendation Score: 76.80
👉 ✅ Highly Recommended

✅ Output saved to: /kaggle/working/final_output.csv
